# Implementación en Raspberry Pi — cambio de dynamic model del GPS

Guía de implementación para el integrante del equipo a cargo de programar,
en la Raspberry Pi, el cambio de perfil dinámico (*dynamic platform model*)
y de tasa de actualización del módulo GPS u-blox durante el vuelo de
**Proyecto Hermes**, disparado por el barómetro (BME280).

Este notebook es la continuación práctica de
[`GPS/Simulacion_GPS.ipynb`](./Simulacion_GPS.ipynb) — ese notebook diseñó y
validó el filtro de Kalman (GPS + IMU) en simulación; este notebook cubre la
parte de firmware/RPi que faltaba: reconfigurar el GPS en vuelo y conectar
esa señal con el filtro.

**No hace falta tener el NEO-6M en la mesa para usar esto**: el protocolo
UBX es el mismo (con pequeñas diferencias de firmware) en el NEO-6M, NEO-7M
y NEO-M8N, así que todo lo de acá se puede probar apenas llegue cualquiera
de los tres, o incluso antes con un mock del puerto serie.


## Respuesta corta a la pregunta del equipo

> *"¿Podría ser que cuando el barómetro cambie sus parámetros eso active
> cierto módulo, y además usar esa modelación para mayor precisión del GPS
> junto con el acelerómetro y giroscopio? ¿El barómetro daría las
> condiciones iniciales para el filtro de Kalman y además los valores para
> activar el cambio de módulo?"*

**Sí, tiene sentido — y de hecho ya estaba anotado como paso pendiente en
`Simulacion_GPS.ipynb`**, en la celda "Próximos pasos antes de producción":

> *"Detección automática solo para la emergencia: [...] conviene detectar el
> reviente en tiempo real (pico sostenido de aceleración vertical negativa,
> o caída brusca de la tasa de ascenso del barómetro BME280)."*
>
> *"Fusionar también el barómetro (BME280): ya está en el proyecto [...] y
> da altitud más estable que el GPS o el acelerómetro integrado en Z — buen
> candidato para otra medición más en `H`, disponible incluso durante el
> corte de GPS."*

Con un matiz importante sobre el término "condiciones iniciales": el
barómetro no resetea literalmente el estado inicial del filtro (`x0` ya sale
del primer fix GPS, ver `simular_escenario`). Lo que sí hace, y que cubre, son **tres cosas separadas**:

1. **Disparador compartido**: la misma lectura de barómetro (velocidad
   vertical sostenida por debajo de un umbral) dispara *a la vez* (a) el
   cambio de `dynModel`/tasa en el GPS por hardware, y (b) el cambio de fase
   que el filtro de Kalman necesita conocer en software. Un solo detector,
   dos consumidores — evita tener la lógica de detección duplicada y
   potencialmente inconsistente en dos lugares.
2. **Medición extra en el filtro** (`H`): el barómetro entra como una fila
   más de medición de altitud/velocidad vertical, igual que ya lo dejó
   anotado el notebook de simulación. Es oro justo durante el corte de GPS
   que se simula en ambos escenarios (`dropout_inicio_s`/`dropout_fin_s`),
   porque no necesita satélites y actualiza mucho más rápido que el GPS.
3. **Aumento temporal de `proceso_var` (Q) en el evento**: cuando el
   detector dispara, conviene inflar el ruido de proceso del filtro por una
   ventana corta, para que reaccione rápido al cambio brusco de dinámica en
   vez de arrastrar el modelo de vuelo anterior. Es, en el fondo, una
   versión simplificada de un solo interruptor de lo que el notebook de
   simulación ya lista como el paso más robusto: el **IMM (Interacting
   Multiple Model)**.

**Ojo con un choque de nombres que puede confundir al equipo**: el "IMM" de
`Simulacion_GPS.ipynb` (varios modelos de dinámica de vuelo compitiendo
dentro del filtro de Kalman, en Python) y el "dynamic model" del GPS u-blox
(`Airborne <1g` / `<4g`, dentro del firmware del chip GPS) sí comparten la
idea de fondo — "decirle al sistema qué tipo de movimiento esperar" — pero
son dos mecanismos completamente distintos, en dos sistemas distintos.
Cambiar el `dynModel` del GPS **no** cambia el modelo del filtro de Kalman
en Python, y viceversa: hay que disparar los dos por separado (aunque desde
el mismo evento detectado).


## Arquitectura de datos

```
                    ┌─────────────────────┐
                    │   BME280 (I2C)       │
                    │   altitud / vz       │
                    └──────────┬───────────┘
                               │  cada lectura
                               ▼
                    ┌─────────────────────┐
                    │  detectar_evento()   │   ← este notebook
                    │  (umbral sostenido)  │
                    └──────────┬───────────┘
                     ¿evento?  │
                 ┌─────────────┴─────────────┐
                 ▼                           ▼
      ┌─────────────────────┐     ┌───────────────────────────┐
      │  GPS u-blox (UART)   │     │  estado de fase           │
      │  UBX CFG-NAV5        │     │  (ASCENSO / EVENTO / VUELO)│
      │  UBX CFG-RATE        │     │  → archivo/socket          │
      └─────────────────────┘     └──────────────┬─────────────┘
                                                    ▼
                                     ┌───────────────────────────┐
                                     │ Filtro de Kalman (Python)  │
                                     │ GPS + IMU + BARO            │
                                     │ (Simulacion_GPS.ipynb        │
                                     │  portado a producción)      │
                                     └───────────────────────────┘
```

Importante: **son dos enlaces serie distintos**, no confundir uno con otro:

- **RPi ↔ módulo GPS** (UART, protocolo UBX/NMEA): para configurar el GPS y
  leer su posición. Es el que se toca en este notebook.
- **RPi ↔ estación en tierra** (protocolo PHUC del proyecto, actualmente
  115200 baudios, ver `computer/communication.py` y `bridge_server.py`):
  para bajar telemetría (`IMU_TYPE`, `GPS_TYPE`, `BARO_TYPE`). Ese firmware
  de vuelo (hoy en el Arduino, pendiente migrar a la RPi según el
  `README.md`) es un programa aparte que arma esos paquetes; el cambio de
  `dynModel` no viaja por ahí.


## Alcance y supuestos

- Todavía no hay un NEO-6M en la mesa (`Simulacion_GPS.ipynb` lo dice
  explícitamente), pero el código de acá sirve igual para NEO-6M, NEO-7M o
  NEO-M8N: el protocolo UBX y los mensajes `CFG-NAV5`/`CFG-RATE` son
  compatibles en los tres.
- El BME280 sí está en el proyecto (aparece en `README.md` y en
  `computer/communication.py` como `BARO_TYPE`).
- Todo el código de este notebook está pensado para **Raspberry Pi con
  Raspberry Pi OS (Linux)**, en C/C++ puro (sin librerías tipo Arduino), tal
  como se pidió.
- El código C de acá se entrega como bloques para copiar a archivos `.c`/`.h`
  y compilar con `gcc` — un notebook de Jupyter no ejecuta C directamente.
  Donde tiene sentido (el cálculo del checksum, para verificarlo antes de
  portarlo) se incluye también una celda de Python **ejecutable**, para
  poder probar la lógica ahí mismo sin hardware.


## Paso 0 — Hardware y conexiones

**RPi ↔ módulo GPS (UART)**

| RPi (GPIO) | Pin físico | GPS |
|---|---|---|
| GPIO14 (TXD) | 8  | RX del GPS |
| GPIO15 (RXD) | 10 | TX del GPS |
| GND           | 6  | GND |
| 3.3V ó 5V (según el módulo — revisar datasheet) | 1 / 2 | VCC |

**RPi ↔ BME280 (I2C)**

| RPi (GPIO) | Pin físico | BME280 |
|---|---|---|
| GPIO2 (SDA) | 3 | SDA |
| GPIO3 (SCL) | 5 | SCL |
| GND          | 9 | GND |
| 3.3V         | 1 | VCC |

En los Raspberry Pi con Bluetooth integrado (3B/3B+/4/Zero W/400), el
Bluetooth usa por defecto el UART "bueno" (PL011, `/dev/ttyAMA0`) y deja al
GPS con un UART por software (`mini UART`) menos estable a baudrates altos.
Para este proyecto conviene liberar el UART bueno para el GPS (ver paso 1).


## Paso 1 — Habilitar el UART para el GPS

```bash
sudo raspi-config
# Interface Options → Serial Port
#   "¿Querés una consola de login por serial?"  → No
#   "¿Querés habilitar el hardware serial port?" → Sí
```

Si el modelo de RPi tiene Bluetooth integrado, para que el GPS quede en el
UART de hardware (`/dev/ttyAMA0`, mapeado como `/dev/serial0`) en vez del
mini UART:

```bash
echo "dtoverlay=disable-bt" | sudo tee -a /boot/firmware/config.txt
sudo systemctl disable hciuart
sudo reboot
```

> Nota: en Raspberry Pi OS más viejos el archivo es `/boot/config.txt` en
> vez de `/boot/firmware/config.txt` — revisar cuál existe antes de editar.

Después del reboot, el código siempre debe abrir `/dev/serial0` (symlink
estable), nunca `/dev/ttyAMA0` ni `/dev/ttyS0` directamente — así el mismo
código sigue andando aunque cambie el modelo de RPi.


## Paso 2 — Habilitar I2C para el barómetro

```bash
sudo raspi-config
# Interface Options → I2C → Sí
sudo reboot

# después del reboot, verificar que el BME280 aparece en el bus:
sudo apt install -y i2c-tools
i2cdetect -y 1
# debería listar 0x76 o 0x77 (dirección típica del BME280)
```


## Paso 3 — Verificar la comunicación antes de programar nada

Con el GPS ya conectado, antes de escribir una sola línea de C conviene
confirmar que manda datos, a qué baudrate, y en qué formato:

```bash
sudo apt install -y minicom
sudo minicom -b 9600 -D /dev/serial0
# (Ctrl-A luego X para salir)
```

Si el módulo está funcionando vas a ver texto tipo `$GPGGA,...` o
`$GNRMC,...` (sentencias NMEA) apareciendo varias veces por segundo. Ese
`9600` es el baudrate de fábrica típico del NEO-6/7/8 — algunos módulos ya
vienen preconfigurados a otro baudrate por el vendedor de la placa; si no
aparece nada legible a 9600, probar 4800, 19200, 38400 o 115200 antes de
sospechar del cableado.


## Paso 4 — El protocolo UBX en profundidad

Todo mensaje UBX tiene esta forma:

| Campo | Tamaño | Contenido |
|---|---|---|
| Sync 1 | 1 byte | `0xB5` (fijo) |
| Sync 2 | 1 byte | `0x62` (fijo) |
| Class  | 1 byte | familia del mensaje (`0x06` = CFG) |
| ID     | 1 byte | mensaje dentro de la familia |
| Length | 2 bytes (little-endian) | tamaño del *payload*, sin contar header ni checksum |
| Payload | `Length` bytes | los datos del mensaje |
| CK_A, CK_B | 2 bytes | checksum de 8 bits (algoritmo Fletcher) sobre Class+ID+Length+Payload |

**Checksum (Fletcher de 8 bits)** — se calcula sobre todo *menos* los dos
bytes de sync:

```
CK_A = 0; CK_B = 0
para cada byte b en [Class, ID, LengthLow, LengthHigh, payload...]:
    CK_A = (CK_A + b) mod 256
    CK_B = (CK_B + CK_A) mod 256
```

### `UBX-CFG-NAV5` (Class `0x06`, ID `0x24`) — cambia el dynamic model

Payload de 36 bytes. Los primeros campos son los que importan acá (el resto
se deja en 0):

| Offset | Campo | Tamaño | Uso |
|---|---|---|---|
| 0 | `mask` | 2 bytes (U2) | qué parámetros aplicar. Bit 0 = `dyn`. Con `mask = 0x0001` el receptor aplica *solo* `dynModel` e ignora el resto del payload (por eso se puede mandar todo lo demás en 0 sin romper nada). |
| 2 | `dynModel` | 1 byte (U1) | `0`=Portable, `6`=Airborne&nbsp;<1g, `7`=Airborne&nbsp;<2g, `8`=Airborne&nbsp;<4g |
| 3–35 | resto | 33 bytes | ignorados si no están marcados en `mask` |

### `UBX-CFG-RATE` (Class `0x06`, ID `0x08`) — cambia la tasa de navegación

Payload de 6 bytes:

| Offset | Campo | Tamaño | Uso |
|---|---|---|---|
| 0 | `measRate` | 2 bytes (U2) | milisegundos entre mediciones (`1000`=1&nbsp;Hz, `100`=10&nbsp;Hz) |
| 2 | `navRate` | 2 bytes (U2) | ciclos por solución de navegación — dejar en `1` |
| 4 | `timeRef` | 2 bytes (U2) | `0`=UTC, `1`=tiempo GPS |


In [1]:
# Referencia ejecutable en Python del checksum y del armado de paquete UBX.
# Sirve para verificar a mano, en el notebook, los bytes exactos que el
# código en C (Paso 5) tiene que producir -- antes de gastar un ciclo de
# compilar/flashear en la Raspberry Pi.

import struct


def ubx_checksum(cuerpo: bytes) -> tuple[int, int]:
    # cuerpo = Class + ID + LengthLow + LengthHigh + Payload (sin los sync)
    ck_a = ck_b = 0
    for b in cuerpo:
        ck_a = (ck_a + b) & 0xFF
        ck_b = (ck_b + ck_a) & 0xFF
    return ck_a, ck_b


def build_ubx_packet(msg_class: int, msg_id: int, payload: bytes) -> bytes:
    length = len(payload)
    cuerpo = bytes([msg_class, msg_id, length & 0xFF, (length >> 8) & 0xFF]) + payload
    ck_a, ck_b = ubx_checksum(cuerpo)
    return bytes([0xB5, 0x62]) + cuerpo + bytes([ck_a, ck_b])


def build_cfg_nav5(dyn_model: int) -> bytes:
    payload = bytearray(36)
    payload[0:2] = struct.pack("<H", 0x0001)  # mask: solo dyn
    payload[2] = dyn_model
    return build_ubx_packet(0x06, 0x24, bytes(payload))


def build_cfg_rate(meas_rate_ms: int, time_ref: int = 1) -> bytes:
    payload = struct.pack("<HHH", meas_rate_ms, 1, time_ref)
    return build_ubx_packet(0x06, 0x08, payload)


# Ejemplos: modo Airborne <1g (fase globo) y Airborne <4g a 10 Hz (post-suelta)
pkt_ascenso = build_cfg_nav5(dyn_model=6)
pkt_vuelo = build_cfg_nav5(dyn_model=8)
pkt_rate_1hz = build_cfg_rate(1000)
pkt_rate_10hz = build_cfg_rate(100)

for nombre, pkt in [
    ("CFG-NAV5 Airborne<1g", pkt_ascenso),
    ("CFG-NAV5 Airborne<4g", pkt_vuelo),
    ("CFG-RATE 1 Hz", pkt_rate_1hz),
    ("CFG-RATE 10 Hz", pkt_rate_10hz),
]:
    print(f"{nombre:24s} ({len(pkt):3d} bytes): {pkt.hex(' ')}")


CFG-NAV5 Airborne<1g     ( 44 bytes): b5 62 06 24 24 00 01 00 06 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 55 b4
CFG-NAV5 Airborne<4g     ( 44 bytes): b5 62 06 24 24 00 01 00 08 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 57 f8
CFG-RATE 1 Hz            ( 14 bytes): b5 62 06 08 06 00 e8 03 01 00 01 00 01 39
CFG-RATE 10 Hz           ( 14 bytes): b5 62 06 08 06 00 64 00 01 00 01 00 7a 12


**Verificación**: el paquete `CFG-RATE 10 Hz` de arriba
(`b5 62 06 08 06 00 64 00 01 00 01 00 7a 12`) coincide byte a byte con un
ejemplo de referencia para "cambiar el GPS u-blox a 10 Hz" publicado de forma
independiente ([Stuart's Projects — Generating UBLOX GPS Configuration
Messages](https://stuartsprojects.github.io/2018/08/26/Generating-UBLOX-GPS-Configuration-Messages.html)),
checksum incluido (`7a 12`). Así que el checksum y el armado de paquete de
esta sección están confirmados contra una fuente externa, no solo contra sí
mismos.


## Paso 5 — Código C: librería mínima UBX (`ubx_gps.h` / `ubx_gps.c`)

Se arma el paquete a mano, byte por byte, en vez de usar un `struct`
empaquetado (`__attribute__((packed))`): con `struct` el resultado depende
sutilmente del compilador/plataforma (alineación, *padding*), y para un
payload de 36 bytes que va a viajar por un cable hacia un chip que no
perdona errores, conviene la construcción explícita aunque sea un poco más
larga de escribir.

```c
// ubx_gps.h
#ifndef UBX_GPS_H
#define UBX_GPS_H

#include <stdint.h>
#include <stddef.h>

// Checksum de 8 bits (Fletcher) usado por UBX. buf = [class, id, lenL, lenH,
// payload...] -- SIN los 2 bytes de sync.
void ubx_checksum(const uint8_t *buf, size_t len, uint8_t *ck_a, uint8_t *ck_b);

// Arma un paquete UBX completo (sync + header + payload + checksum) en out.
// out debe tener espacio para 8 + payload_len bytes. Devuelve el tamaño total.
size_t ubx_build_packet(uint8_t msg_class, uint8_t msg_id,
                         const uint8_t *payload, uint16_t payload_len,
                         uint8_t *out);

// dyn_model: 0=Portable, 6=Airborne<1g, 7=Airborne<2g, 8=Airborne<4g
int ubx_set_dynamic_model(int fd, uint8_t dyn_model);

// meas_rate_ms: milisegundos entre fixes (1000 = 1 Hz, 100 = 10 Hz)
int ubx_set_nav_rate(int fd, uint16_t meas_rate_ms);

#endif // UBX_GPS_H
```

```c
// ubx_gps.c
#include "ubx_gps.h"
#include <unistd.h>
#include <string.h>

void ubx_checksum(const uint8_t *buf, size_t len, uint8_t *ck_a, uint8_t *ck_b) {
    uint8_t a = 0, b = 0;
    for (size_t i = 0; i < len; i++) {
        a = (uint8_t)(a + buf[i]);
        b = (uint8_t)(b + a);
    }
    *ck_a = a;
    *ck_b = b;
}

size_t ubx_build_packet(uint8_t msg_class, uint8_t msg_id,
                         const uint8_t *payload, uint16_t payload_len,
                         uint8_t *out) {
    size_t i = 0;
    out[i++] = 0xB5;
    out[i++] = 0x62;
    out[i++] = msg_class;
    out[i++] = msg_id;
    out[i++] = (uint8_t)(payload_len & 0xFF);
    out[i++] = (uint8_t)((payload_len >> 8) & 0xFF);
    memcpy(out + i, payload, payload_len);
    i += payload_len;

    uint8_t ck_a, ck_b;
    // el checksum cubre class+id+length+payload, es decir todo lo escrito
    // después de los 2 bytes de sync (por eso "out + 2").
    ubx_checksum(out + 2, i - 2, &ck_a, &ck_b);
    out[i++] = ck_a;
    out[i++] = ck_b;
    return i;
}

int ubx_set_dynamic_model(int fd, uint8_t dyn_model) {
    uint8_t payload[36] = {0};   // todo en 0 = "no aplicar / no importa"
    payload[0] = 0x01;           // mask byte bajo -> bit0 (dyn) = 1
    payload[1] = 0x00;           // mask byte alto
    payload[2] = dyn_model;      // offset 2: dynModel
    // offsets 3..35 quedan en 0: el receptor los ignora porque mask
    // solo pide aplicar dynModel.

    uint8_t packet[8 + 36];
    size_t n = ubx_build_packet(0x06, 0x24, payload, sizeof(payload), packet);
    return (write(fd, packet, n) == (ssize_t)n) ? 0 : -1;
}

int ubx_set_nav_rate(int fd, uint16_t meas_rate_ms) {
    uint8_t payload[6];
    payload[0] = (uint8_t)(meas_rate_ms & 0xFF);
    payload[1] = (uint8_t)((meas_rate_ms >> 8) & 0xFF);
    payload[2] = 0x01; payload[3] = 0x00;   // navRate = 1 (fijo)
    payload[4] = 0x01; payload[5] = 0x00;   // timeRef = 1 (tiempo GPS)

    uint8_t packet[8 + 6];
    size_t n = ubx_build_packet(0x06, 0x08, payload, sizeof(payload), packet);
    return (write(fd, packet, n) == (ssize_t)n) ? 0 : -1;
}
```

Y la apertura del puerto serie hacia el GPS (recordar: **este `/dev/serial0`
es el enlace RPi↔GPS, no tiene nada que ver con el puerto que usa
`bridge_server.py` hacia la estación en tierra**):

```c
// serial_gps.c
#include <fcntl.h>
#include <termios.h>
#include <unistd.h>

int abrir_puerto_gps(const char *device) {
    int fd = open(device, O_RDWR | O_NOCTTY | O_SYNC);
    if (fd < 0) return -1;

    struct termios tty;
    tcgetattr(fd, &tty);
    cfmakeraw(&tty);
    cfsetispeed(&tty, B9600);   // baud de fábrica -- confirmar en banco (Paso 3)
    cfsetospeed(&tty, B9600);
    tty.c_cflag |= (CLOCAL | CREAD);
    tty.c_cflag &= ~PARENB;     // sin paridad
    tty.c_cflag &= ~CSTOPB;     // 1 bit de stop
    tty.c_cflag &= ~CSIZE;
    tty.c_cflag |= CS8;         // 8 bits de datos
    tcsetattr(fd, TCSANOW, &tty);

    return fd;
}
```


## Paso 6 — Lectura del barómetro (BME280) en C

Acá conviene una advertencia honesta: el BME280 no entrega presión/altitud
en unidades directas, entrega registros crudos que hay que combinar con ~10
coeficientes de calibración propios del chip usando fórmulas de punto fijo
específicas de Bosch. Reimplementar esa compensación a mano desde cero es la
fuente número uno de bugs silenciosos (altitudes que "casi" cierran pero
están corridas) en este tipo de proyecto estudiantil — no aporta nada
reimplementarlo, y si hay un error sutil es difícil de encontrar después.

Recomendación: usar el driver oficial de Bosch en C
([`BoschSensortec/BME280_driver`](https://github.com/boschsensortec/BME280_driver)
en GitHub, license MIT) para la lectura/calibración, y escribir solo la capa
fina de arriba con la lógica de este proyecto:

```c
// baro_interface.h — capa fina sobre BME280_driver (Bosch, oficial)
double baro_leer_altitud_m(void);
double baro_leer_velocidad_vertical_ms(void);  // derivada filtrada de la altitud
```

`baro_leer_velocidad_vertical_ms` se puede implementar simplemente como una
diferencia finita entre dos lecturas de altitud sucesivas dividida por el
`dt` entre ellas, pasada por un filtro pasa-bajos de un polo (una sola
línea: `vz_filtrada += alpha * (vz_nueva - vz_filtrada)`) para no disparar
el detector del Paso 7 por ruido de una sola lectura.


## Paso 7 — Lógica de disparo del evento

Implementa literalmente lo que ya había quedado anotado en
`Simulacion_GPS.ipynb`: exigir que la condición se sostenga varias lecturas
seguidas, no una sola, para no disparar por una ráfaga de ruido del sensor.

```c
// evento_separacion.h
#include <stdbool.h>

typedef enum { FASE_ASCENSO, FASE_EVENTO, FASE_VUELO } fase_t;

#define UMBRAL_VZ_MS       -3.0   // m/s -- ILUSTRATIVO, ajustar con datos reales
#define MUESTRAS_SOSTENIDO  5     // lecturas seguidas que deben cumplir el umbral

// Devuelve true la vez que se cumple la condición sostenida (flanco, no nivel).
bool detectar_evento(double vz_actual, int *contador) {
    if (vz_actual < UMBRAL_VZ_MS) {
        (*contador)++;
    } else {
        *contador = 0;
    }
    return (*contador == MUESTRAS_SOSTENIDO);  // == y no >= : dispara una sola vez
}
```

`UMBRAL_VZ_MS` y `MUESTRAS_SOSTENIDO` son placeholder, igual que el resto de
los parámetros ilustrativos del notebook de simulación (`T_REVIENTA`,
`T_CAIDA_LIBRE`, etc.) — hay que calibrarlos con datos reales de banco
apenas estén el BME280 y el perfil de vuelo definidos, siguiendo el mismo
criterio que ya se usó ahí ("pico sostenido de aceleración vertical
negativa, o caída brusca de la tasa de ascenso").


## Paso 8 — Programa principal (`gps_dynmodel_switch.c`)

Junta todo: fase inicial de ascenso (Airborne&nbsp;<1g, 1&nbsp;Hz), monitoreo
del barómetro, y al detectar el evento sostenido pasa el GPS a
Airborne&nbsp;<4g a 10&nbsp;Hz y avisa la fase nueva para que el filtro de
Kalman (Paso 9) la lea.

```c
// gps_dynmodel_switch.c
#include <stdio.h>
#include <unistd.h>
#include "ubx_gps.h"
#include "evento_separacion.h"

int abrir_puerto_gps(const char *device);              // serial_gps.c
double baro_leer_velocidad_vertical_ms(void);           // baro_interface.c

static void escribir_estado_para_kalman(fase_t fase) {
    // El proceso de Python que corre el filtro de Kalman en vuelo (versión
    // de producción de crear_filtro_ca3d_bias / simular_escenario, ver
    // "Próximos pasos" de Simulacion_GPS.ipynb) lee este archivo para saber
    // cuándo: (a) sumar la medición de altímetro barométrico a H, y
    // (b) inflar Q por la ventana de transición.
    // Para arrancar alcanza un archivo de texto plano; si hace falta más
    // velocidad se cambia por un socket UNIX o una FIFO sin tocar el resto.
    FILE *f = fopen("/tmp/hermes_fase.txt", "w");
    if (!f) return;
    fprintf(f, "%d\n", (int)fase);
    fclose(f);
}

int main(void) {
    int fd = abrir_puerto_gps("/dev/serial0");
    if (fd < 0) { perror("no se pudo abrir el GPS"); return 1; }

    // --- Configuración inicial: fase de ascenso del globo ---
    ubx_set_dynamic_model(fd, 6 /* Airborne <1g */);
    ubx_set_nav_rate(fd, 1000 /* 1 Hz */);
    fase_t fase = FASE_ASCENSO;
    escribir_estado_para_kalman(fase);

    int contador = 0;

    while (1) {
        double vz = baro_leer_velocidad_vertical_ms();

        if (fase == FASE_ASCENSO && detectar_evento(vz, &contador)) {
            fase = FASE_EVENTO;
            escribir_estado_para_kalman(fase);

            ubx_set_dynamic_model(fd, 8 /* Airborne <4g */);
            ubx_set_nav_rate(fd, 100 /* 10 Hz */);

            fase = FASE_VUELO;
            escribir_estado_para_kalman(fase);
        }

        usleep(40000);  // ~25 Hz, ritmo típico de lectura de un BME280
    }
}
```

**Compilar:**

```bash
gcc -Wall -O2 -o gps_dynmodel_switch \
    gps_dynmodel_switch.c ubx_gps.c serial_gps.c evento_separacion.c baro_interface.c \
    -lbme280   # o el nombre que tenga la lib del driver de Bosch una vez linkeado

# Acceso al puerto serie sin sudo:
sudo usermod -aG dialout $USER   # requiere reloguear para que tome efecto
```


## Paso 9 — Cómo lo lee el lado de Kalman (Python)

Del lado del filtro (hoy en `Simulacion_GPS.ipynb`, mañana portado a
`computer/kalman.py` según el propio "Próximos pasos" de ese notebook), leer
la fase es tan simple como:

```python
def leer_fase():
    try:
        with open("/tmp/hermes_fase.txt") as f:
            return int(f.read().strip())
    except (FileNotFoundError, ValueError):
        return 0  # FASE_ASCENSO por defecto
```

Y usarla en el loop del filtro para las dos cosas que se explicaron en la
respuesta corta de arriba:

```python
fase_anterior = None
for i, t in enumerate(t_imu):
    fase = leer_fase()

    if fase != fase_anterior:
        # Transición de fase detectada: inflar el ruido de proceso por una
        # ventana corta para que el filtro no arrastre el modelo de vuelo
        # anterior (versión simple del IMM que ya está en "Próximos pasos").
        kf.Q *= 5.0
        fase_anterior = fase

    kf.predict(u=...)
    if hay_fix_gps(i):
        kf.update(z=..., R=...)
    if hay_lectura_baro(i):
        # Medición extra de H: altitud barométrica, disponible incluso
        # durante el corte de GPS -- exactamente lo que pedía el notebook
        # de simulación.
        kf.update(z=altitud_baro[i], R=SIGMA_BARO, H=H_solo_altitud)
```

(El `kf.Q *= 5.0` y `SIGMA_BARO` son ilustrativos, mismo criterio que el
resto de parámetros del proyecto: se ajustan con datos reales de banco.)


## Checklist de pruebas en banco antes de integrar al avión

- [ ] Verificar con `minicom` que el GPS responde y a qué baudrate (Paso 3).
- [ ] Enviar `ubx_set_dynamic_model` y `ubx_set_nav_rate` "a mano" (un
      programita chico que solo mande esos dos comandos) y confirmar con
      `u-center` (en una PC) o leyendo las tramas UBX de respuesta que el
      cambio se aplicó.
- [ ] Confirmar la dirección I2C real del BME280 con `i2cdetect` (Paso 2) —
      puede ser `0x76` o `0x77` según el módulo.
- [ ] Calibrar `UMBRAL_VZ_MS` y `MUESTRAS_SOSTENIDO` con una prueba real de
      caída (por ejemplo soltando el sensor desde una altura conocida) antes
      de confiar en el valor ilustrativo.
- [ ] Confirmar que `/tmp/hermes_fase.txt` (o el mecanismo que lo reemplace)
      se lee correctamente desde el proceso de Python mientras el de C sigue
      escribiendo — probar los dos procesos corriendo juntos en la RPi antes
      del día del vuelo.
- [ ] Una vez migrado el firmware de vuelo del Arduino a la RPi (pendiente
      en el `README.md`), confirmar que este programa y el que arma los
      paquetes PHUC (`GPS_TYPE`, `BARO_TYPE`) no compiten por el mismo
      puerto serie del GPS.


## Referencias

- [u-blox NEO-6 Data Sheet](https://content.u-blox.com/sites/default/files/products/documents/NEO-6_DataSheet_(GPS.G6-HW-09005).pdf)
- [u-blox NEO-7 Product Summary](https://content.u-blox.com/sites/default/files/products/documents/NEO-7_ProductSummary_(UBX-13003342).pdf)
- [u-blox NEO-M8 Product Summary](https://content.u-blox.com/sites/default/files/products/documents/NEO-M8_ProductSummary_UBX-16000345.pdf)
- [UKHAS Wiki — GPS Modules (límites COCOM)](https://ukhas.org.uk/doku.php?id=guides:gps_modules)
- [Ava HAB Project — Setting Airborne Dynamic Model](https://ava.upuaut.net/?p=750)
- [Bosch BME280_driver (oficial, C)](https://github.com/boschsensortec/BME280_driver)
- [Stuart's Projects — Generating UBLOX GPS Configuration Messages](https://stuartsprojects.github.io/2018/08/26/Generating-UBLOX-GPS-Configuration-Messages.html)
- [`GPS/Simulacion_GPS.ipynb`](./Simulacion_GPS.ipynb) — filtro de Kalman GPS+IMU de Proyecto Hermes
- [`README.md`](../README.md) y [`computer/communication.py`](../computer/communication.py) — protocolo PHUC del proyecto
